# Vestibular-BR — avaliação (vLLM)

GPU T4 x2.

In [ ]:
GITHUB_REPO = "https://github.com/MariaEduarda-jg/Text-to-SQL.git"
HF_USER = "MariaEduarda-jg"

import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for k in ["HF_TOKEN", "GEMINI_API_KEY"]:
    try:
        os.environ[k] = secrets.get_secret(k)
    except Exception:
        pass

In [ ]:
!git clone -q $GITHUB_REPO repo
%cd repo
!pip install -q -e . vllm
!vbr data

## Escala: Qwen3.5 0.8B → 9B

In [ ]:
for m in ["Qwen/Qwen3.5-0.8B", "Qwen/Qwen3.5-2B", "Qwen/Qwen3.5-4B"]:
    !vbr eval -b test -m vllm:{m}
!VLLM_TP=2 vbr eval -b test -m vllm:Qwen/Qwen3.5-9B

## Imagem vs. descrição vs. sem imagem

In [ ]:
for mode in ["image", "blind"]:
    !vbr eval -b enem -m vllm:Qwen/Qwen3.5-2B --mode {mode}
    !VLLM_TP=2 vbr eval -b enem -m vllm:Qwen/Qwen3.5-9B --mode {mode}

## Contaminação: alternativas embaralhadas

In [ ]:
!VLLM_TP=2 vbr contamination -b bluex -m vllm:Qwen/Qwen3.5-9B --seeds 1 --seeds 2

## pass@4 no treino (filtro de dificuldade do GRPO)

In [ ]:
!vbr eval -b train -m vllm:Qwen/Qwen3.5-2B -n 4 --temperature 1.0 --max-tokens 1024 --tag pass4

## Modelos treinados

In [ ]:
for tag in ["sft", "grpo", "sft-grpo"]:
    !vbr eval -b test -m vllm:Qwen/Qwen3.5-2B --lora {HF_USER}/vestibular-br-2b-{tag} --tag {tag}

In [ ]:
!vbr report
!zip -qr /kaggle/working/resultados.zip results